[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_19/MFM_ch19_integrated_romania/MFM_ch19_integrated_romania.ipynb)

# MFM_ch19_integrated_romania

Integrated case study for an investor who measures wealth in lei and holds 50% BET and 50% Euro Stoxx 50 (daily closes and the BNR EUR/RON reference rate, 2005-2026): Lo-MacKinlay variance ratios and the Chow-Denning test (Chapter 2), weekly factor exposures with Newey-West errors (Chapter 3), GARCH(1,1)-t volatility (Chapter 5), the rolling three-year minimum-variance currency hedge ratio (Chapter 6) and the weekly VaR 1% and ES 2.5% by historical simulation for no hedge, the rolling hedge and a full hedge (Chapter 7).

*Modelling Financial Markets (MFM), Daniel Traian Pele, Bucharest University of Economic Studies*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_19'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
# Colab: !pip install arch
import os
import numpy as np
import pandas as pd
try:
    import arch  # noqa: F401
except ModuleNotFoundError as exc:
    if exc.name != 'arch':
        raise
    import subprocess
    import sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'arch'], check=True)
from IPython.display import Image, display
import warnings
warnings.filterwarnings('ignore')


## The code of the chapters

`integrated_cases.py` imports the modules of each chapter from its own folder and never writes the charts or results of another chapter.

In [ ]:
# The case study calls the code of several chapters of the course (Quantlets/Ch_NN). Inside a clone of the
# course repository the local files are used; otherwise the modules are downloaded from GitHub.
import sys
import urllib.request
RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/'
FILES = ["slide_fit.py", "slide_fit_sizes.json", "Ch_19/integrated_cases.py", "Ch_19/inference19.py", "Ch_02/eff_tests.py", "Ch_02/mfm_data.py", "Ch_03/mfm_data.py", "Ch_05/garch_tools.py", "Ch_07/risk_measures.py"]
QL = os.environ.get('MFM_QUANTLETS') or os.path.abspath(os.path.join('..', '..'))
if not os.path.exists(os.path.join(QL, 'Ch_19', 'integrated_cases.py')):
    QL = os.path.abspath('mfm_quantlets')
    for f in FILES:
        path = os.path.join(QL, f)
        os.makedirs(os.path.dirname(path), exist_ok=True)
        if not os.path.exists(path):
            urllib.request.urlretrieve(RAW + f, path)
sys.path.insert(0, os.path.join(QL, 'Ch_19'))
os.environ['MFM_CHART_DIR'] = OUT_DIR     # the chart (PDF, PNG) goes to OUT_DIR
os.environ['MFM_OUT_DIR'] = OUT_DIR       # and so do the CSV series
import integrated_cases as IC


## Steps 1-3, Chapters 2, 3 and 5: random walk tests, exposures, volatility

Variance ratios with the heteroskedasticity-robust statistic on daily returns; exposures on weekly (Friday) returns, because the BNR fixes the rate at 13:00 and the exchanges close at different times.

## Steps 4-5, Chapters 6 and 7: the hedge and its tail risk

The minimum-variance hedge ratio h* = Cov(r_RON, f) / Var(f) on a rolling three-year window, known at the start of each week; weekly ES 2.5% of the 50/50 portfolio for h = 0, the rolling h* and h = 1.

In [ ]:
R, D = IC.romania_case()
IC.fig_romania(R, D)
display(Image(os.path.join(OUT_DIR, 'ch19_int_romania.png')))

## The numbers behind the chart

In [ ]:
pd.DataFrame({k: {'z*(2)': v['z']['2'], 'z*(5)': v['z']['5'], 'Chow-Denning p': v['cd_p']} for k, v in R['step1'].items()}).T.round(3)

In [ ]:
print('Hedge ratio:', {k: round(v, 3) if isinstance(v, float) else v for k, v in R['step4'].items()})
pd.DataFrame({p: {k: v['es'] for k, v in R['step5'][p].items()} for p in R['step5']}).round(2)

Figure file: [ch19_int_romania](./ch19_int_romania.png). The analysis displays its freshly generated figure inline and saves PDF and PNG files in OUT_DIR.